In [1]:
import os
import sys
import redis

# Cancella tutti i record del "chat store" in Redis.
# Usa una connessione Redis già definita in altre celle (redis_client, r, redis_conn) se presente,
# altrimenti si connette a REDIS_URL o localhost:6379.

# Recupera client Redis esistente se definito nelle celle sottostanti
redis_client = globals().get("redis_client") or globals().get("r") or globals().get("redis_conn")

if redis_client is None:
    redis_url = os.getenv("REDIS_URL", "redis://localhost:6379/0")
    redis_client = redis.from_url(redis_url)

# Pattern/chiavi tipiche per uno "chat store" — modifica se necessario.
patterns = ["chat:*", "chat_store", "chats:*", "chatstore"]

# Raccoglie tutte le chiavi corrispondenti ai pattern
keys_to_delete = set()
for p in patterns:
    if "*" in p:
        for k in redis_client.scan_iter(match=p):
            keys_to_delete.add(k if isinstance(k, str) else k.decode())
    else:
        if redis_client.exists(p):
            keys_to_delete.add(p)

if not keys_to_delete:
    print("Nessuna chiave del chat store trovata per i pattern:", patterns)
    sys.exit(0)

print(f"Trovate {len(keys_to_delete)} chiavi da cancellare (esempi):", list(keys_to_delete)[:10])
confirm = input("Confermi la cancellazione di tutte queste chiavi? Digita 'yes' per procedere: ")
if confirm.strip().lower() != "yes":
    print("Operazione annullata.")
    sys.exit(0)

# Cancella le chiavi (usa UNLINK se disponibile)
try:
    unlink = getattr(redis_client, "unlink", None)
    if unlink:
        # Unlink accetta più argomenti, quindi inoltriamo in chunk
        keys_list = list(keys_to_delete)
        chunk_size = 500
        for i in range(0, len(keys_list), chunk_size):
            unlink(*keys_list[i : i + chunk_size])
    else:
        # fallback a delete
        redis_client.delete(*list(keys_to_delete))
    print(f"Cancellate {len(keys_to_delete)} chiavi.")
except Exception as e:
    print("Errore durante la cancellazione:", e)
    raise

Trovate 22 chiavi da cancellare (esempi): ['chat:8293d561f9b6ed09f3e6ee2f5622c143f122e21d769eb24febdd52481245833b', 'chat:8674bcccb0f34208886ad1811c048779a73226d982c1ee71214ee20d7df3ce4c', 'chat:f9532b9476c72a1a7e110129cec83ac4d9bdf21086586336e7dc9e216bb67925', 'chat:b05e4282288d8d0238948c45ab17b372df08c8fa2a4fdffe182c6fa7354333bd', 'chat:644cfe8616d1071ee7d36dfd9b217ec8de9f25711f2c56bbfcd558f9a63227dc', 'chat:800845a992f8820f41786bad16c7dbca87c940622ad7b7449424881f68d802a0', 'chat:21368fc80b30c72a281279d1405b5574d53c9da243b06d854b5fda75ac7e1585', 'chat:1d8ea1447676229eb81927b47a61ccd1daf155e75baf8f1e64ce286a68fa430a', 'chat:031d40be1aa49ae411b79048842a22f3adc67e5b2fd93760bac771646af146b5', 'chat:20fdea1b497723a72bfb161ff9018f91183fb587fda0b10a9d90fb9795d29deb']
Cancellate 22 chiavi.
